# bottleneck-latent-projection composite — cx26: AE forward: flatten → encode → bottleneck z → decode → MSE recon loss

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `mse-reconstruction-loss`, `bottleneck-latent-projection`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "bottleneck-latent-projection"
DD_ATOM_IDS = ["mse-reconstruction-loss", "bottleneck-latent-projection"]
DD_SUBTOPICS = ["Generative: MSE reconstruction loss", "Generative: Bottleneck latent projection"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An autoencoder squeezes the input through a low-dim BOTTLENECK `z`, then reconstructs. The reconstruction quality is measured by MSE. Two atoms wire together:

1. **bottleneck-latent-projection** — the encoder ends in a Linear layer that maps the feature dimension down to `latent_dim` (e.g. 784 → 2 for MNIST visualisation). This is the INFORMATION BOTTLENECK: the model can only express what fits through `z`.
2. **mse-reconstruction-loss** — `F.mse_loss(x_hat, x)` measures how much information made it back. A wider bottleneck → lower MSE (less information lost); a narrower bottleneck → higher MSE.

**Anatomy of `AE.forward`.**
```python
def forward(self, x):
    z = self.encoder(x)                 # bottleneck-latent-projection: shape (B, latent_dim).
    x_hat = self.decoder(z)             # decode back to input shape.
    return x_hat, z

x_hat, z = model(x)
loss = F.mse_loss(x_hat, x)             # mse-reconstruction-loss.
```

**Why test together.** The bottleneck shape and recon loss are tightly coupled: if you mess up `latent_dim` or forget to map back through the decoder, the recon loss either explodes or stops being a meaningful signal.

### Composite Exercise — AE forward: flatten → encode → bottleneck z → decode → MSE recon loss

**Atoms exercised together**: `mse-reconstruction-loss`, `bottleneck-latent-projection`

Implement `cx26_make_autoencoder(input_dim, hidden_dim, latent_dim)` which returns a `SimpleAE` instance (a `nn.Module`).

Required structure:
- `SimpleAE.__init__(self, input_dim, hidden_dim, latent_dim)`:
  - `self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim))` — the final Linear is the BOTTLENECK (atom: bottleneck-latent-projection).
  - `self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))`.
  - `self.latent_dim = latent_dim` (so the test can read it back).
- `SimpleAE.forward(self, x)` — returns `(x_hat, z)`:
  - `z = self.encoder(x)` — shape `(B, latent_dim)`.
  - `x_hat = self.decoder(z)` — shape `(B, input_dim)`, matches `x`.

Then also: the test calls `F.mse_loss(x_hat, x)` (atom: mse-reconstruction-loss) on the forward output and verifies the scalar is sensible.

The test verifies:
- `z` has shape `(B, latent_dim)` — the bottleneck actually squeezes.
- `x_hat` has the same shape as `x`.
- `F.mse_loss(x_hat, x)` is a finite non-negative scalar.
- A wider bottleneck (latent_dim=16) achieves LOWER recon loss after a few training steps than a narrower bottleneck (latent_dim=2). This is the information-bottleneck consequence.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx26_make_autoencoder(input_dim, hidden_dim, latent_dim):
    """Return a SimpleAE nn.Module with the bottleneck encoder + symmetric decoder."""
    raise NotImplementedError

def _test_cx26():
    t.manual_seed(0)
    model = cx26_make_autoencoder(input_dim=8, hidden_dim=16, latent_dim=2)
    assert isinstance(model, nn.Module), 'must return an nn.Module instance'
    assert hasattr(model, 'encoder') and hasattr(model, 'decoder'), 'need encoder + decoder attrs'
    assert getattr(model, 'latent_dim', None) == 2, 'latent_dim attribute must be set on the module'

    # Case A: forward returns (x_hat, z) with correct shapes.
    x = t.randn(5, 8)
    out = model(x)
    assert isinstance(out, tuple) and len(out) == 2, 'forward must return (x_hat, z) tuple'
    x_hat, z = out
    assert z.shape == (5, 2), f'z bottleneck shape must be (B, latent_dim)=(5,2); got {tuple(z.shape)}'
    assert x_hat.shape == x.shape, f'x_hat shape mismatch: {tuple(x_hat.shape)} vs {tuple(x.shape)}'

    # Case B: MSE recon loss is a finite scalar.
    loss = F.mse_loss(x_hat, x)
    assert loss.ndim == 0, 'F.mse_loss must give scalar by default'
    assert t.isfinite(loss).item(), 'loss must be finite'
    assert loss.item() >= 0.0, 'MSE non-negative'

    # Case C: wider bottleneck → lower recon loss after a few steps of training.
    def _train_and_eval(latent_dim, n_steps=80):
        t.manual_seed(0)
        m = cx26_make_autoencoder(input_dim=8, hidden_dim=16, latent_dim=latent_dim)
        opt = t.optim.Adam(m.parameters(), lr=5e-3)
        t.manual_seed(123)
        data = t.randn(64, 8)
        for _ in range(n_steps):
            x_hat, _ = m(data)
            loss = F.mse_loss(x_hat, data)
            opt.zero_grad(); loss.backward(); opt.step()
        with t.no_grad():
            x_hat, _ = m(data)
            return F.mse_loss(x_hat, data).item()

    loss_narrow = _train_and_eval(latent_dim=2)
    loss_wide = _train_and_eval(latent_dim=16)
    assert loss_wide < loss_narrow, (
        f'wider bottleneck (latent_dim=16) should beat narrower (latent_dim=2); '
        f'got wide={loss_wide:.4f} narrow={loss_narrow:.4f}'
    )
    _dd_passed.add('cx26')

_test_cx26()

<details><summary>Show solution — cx26</summary>

```python
def cx26_make_autoencoder(input_dim, hidden_dim, latent_dim):
    class SimpleAE(nn.Module):
        def __init__(self, input_dim, hidden_dim, latent_dim):
            super().__init__()
            # Atom A (bottleneck-latent-projection): encoder ends at latent_dim.
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, latent_dim),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
            )
            self.latent_dim = latent_dim

        def forward(self, x):
            z = self.encoder(x)
            x_hat = self.decoder(z)
            return x_hat, z

    return SimpleAE(input_dim, hidden_dim, latent_dim)
```

Returning `(x_hat, z)` (rather than only `x_hat`) is the ARENA convention — you almost always want the latent for visualisation, anomaly detection, or downstream tasks. The loss is computed on `x_hat` only; `z` is the bottleneck representation.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["Generative: MSE reconstruction loss", "Generative: Bottleneck latent projection"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()